# CS781 Phase 3 — Notebook P3_02: Rhetorical-role prediction with the Phase-1 models

A **hierarchical** model: the Phase-1 Transformer reads windows of consecutive sentences (token level), each
sentence becomes one vector, and a small sequence model over the sentence vectors predicts one of the 13 roles per
sentence. One notebook runs a queue of experiments (one per GPU) and writes, for each, dev scores, an error
analysis and a ready `submission.zip`.

| Step | What happens |
|---|---|
| 1 | config: Phase-1 weights, experiment list, which queue to run |
| 2 | inputs: P3_01's documents (sentence token ids, labels) + the Phase-1 checkpoints |
| 3 | model classes (unchanged from NB3) |
| 4 | backbone (Phase-1 model as token encoder, optional soft prompt) |
| 5 | sentence model: pooling → (+ position, domain) → BiLSTM over sentences → linear or CRF |
| 6 | adaptation methods: full, frozen, LoRA, **rsLoRA**, **LoRMA** (+, π), **IA³**, **prompt tuning**, BitFit, adapters |
| 7 | chunks of sentences → windows of tokens; class weights |
| 8 | inference over overlapping chunks, decoding, the official metric, dev tuning, error analysis |
| 9 | training loop (+ knowledge distillation), experiment runner, submission |
| 10 | unit tests (CPU, seconds) |
| 11 | run the queue on all GPUs | 12 | results table |

**Kaggle settings**: Accelerator **GPU T4 x2**, Internet not needed.
**Inputs**: (1) the **P3_01 output** (`p3/data/docs.pkl`), (2) the **Phase-1 weight datasets** (A0, A1, B, E, F —
names editable in step 1), (3) *optional* earlier versions of this notebook (finished experiments are skipped;
a KD teacher's weights are found there).

In [1]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1148.8 GB / total 8758.4 GB
HF token found: True


## 1. Configuration

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F, pickle, zipfile, threading, queue as _queue, traceback, itertools
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support, confusion_matrix
try: display
except NameError: display = print

P3_SMOKE = os.environ.get("CS781_P3_SMOKE") == "1"      # local pipeline test only (tiny models)

P1_SOURCES = {
    "A0": dict(dataset="cs781-p1-a0", run_name="A0_gpt2_110M_SC80_epoch_1000M"),
    "A1": dict(dataset="cs781-p1-a1", run_name="A1_modern_110M_SC80_epoch_1000M"),
    "B":  dict(dataset="cs781-p1-b",  run_name="B_encdec_6x6_SC80_epoch_1000M"),
    "E":  dict(dataset="cs781-p1-e",  run_name="E_prefixlm_110M_SC80_epoch_1000M"),
    "F":  dict(dataset="cs781-p1-f",  run_name="F_localglobal_gqa_alibi_SC80_epoch_1000M"),
}

DEFAULTS = dict(
    model="E", attn="bidir", init="pretrained",
    method="full", lr=None,
    lora_r=16, lora_alpha=32, lora_targets="all", lorma_variant="plus", topk=4, adapter_dim=64, prompt_len=16,
    # ---- context ----
    ctx="window",           # "sentence": each sentence encoded alone | "window": consecutive sentences share a token window
    win_len=512,            # tokens per window (sentences are packed whole into windows)
    max_sent_tok=128,       # longer sentences are truncated (1.9% of sentences)
    chunk_sents=48,         # sentences per training/inference chunk (the sentence-level model sees this many)
    pool="mean",            # sentence vector: "mean" of its tokens | "sep" = state at its end-of-sentence separator
    sent_lstm=256,          # >0: BiLSTM over the sentence vectors of a chunk (hidden per direction); 0 = none
    pos_emb=True,           # add an embedding of the sentence's relative position in the document (32 buckets)
    domain_emb=True,        # add a CL / IT embedding
    head="linear",          # "linear" | "crf" (transitions between the 13 roles)
    domains=("CL", "IT"),   # train/evaluate on these domains only (per-domain models)
    head_dropout=0.1, backbone_dropout=0.1,
    epochs=12, batch=2, lr_head=1e-3, weight_decay=0.01, warmup=0.06, grad_clip=1.0,
    class_weight="sqrt_inv", tune_bias=True,
    final="dev_model",      # "dev_model" | "retrain_all" (train + dev documents, best dev epoch count)
    teacher=None, kd_alpha=0.5, kd_T=2.0,
    seed=0,
)
METHOD_LR = {"full": 5e-5, "frozen": 0.0, "topk": 1e-4, "lora": 5e-4, "rslora": 5e-4, "lorma": 5e-4, "ia3": 3e-3,
             "prompt": 3e-2, "bitfit": 1e-3, "adapter": 1e-3}

EXPERIMENTS = {
    # ---- Stage A: context ladder (E, full fine-tuning) + transfer control ----
    "a1_E_sentence_only":   dict(ctx="sentence", sent_lstm=0, pos_emb=False, domain_emb=False),
    "a2_E_window":          dict(sent_lstm=0, pos_emb=False, domain_emb=False),
    "a3_E_window_lstm_pos": dict(),                                   # = DEFAULTS (window + BiLSTM + position + domain)
    "a4_E_window_lstm_crf": dict(head="crf"),
    "a5_E_CL_only":         dict(domains=("CL",)),
    "a6_E_IT_only":         dict(domains=("IT",)),
    "a7_A1_random_init":    dict(model="A1", attn="causal", init="random", lr=3e-4),
    # ---- Stage B: Phase-1 backbones (full fine-tuning, best context set-up) ----
    "b1_A0_causal":         dict(model="A0", attn="causal"),
    "b2_A1_causal":         dict(model="A1", attn="causal"),
    "b3_F_causal":          dict(model="F", attn="causal"),
    "b4_B_encoder":         dict(model="B"),
    "b5_A1_causal_seppool": dict(model="A1", attn="causal", pool="sep"),
    # ---- Stage C: adaptation techniques on E ----
    "c1_E_frozen":          dict(method="frozen", epochs=20),
    "c2_E_lora":            dict(method="lora", epochs=20),
    "c3_E_rslora":          dict(method="rslora", epochs=20),
    "c4_E_lorma":           dict(method="lorma", epochs=20),
    "c5_E_lorma_pi":        dict(method="lorma", lorma_variant="pi", lora_targets="attn", epochs=20),
    "c6_E_ia3":             dict(method="ia3", epochs=20),
    "c7_E_prompt":          dict(method="prompt", epochs=20),
}
# ---- FINAL: one Phase-1 model + one technique from the Phase-3 list each, retrained on train + dev ----
EXPERIMENTS.update({
    "f1_E_lorma":          dict(model="E", method="lorma", epochs=20, final="retrain_all"),
    "f2_A1_seppool_lora":  dict(model="A1", attn="causal", pool="sep", method="lora", epochs=20, final="retrain_all"),
    "f3_F_seppool_ia3":    dict(model="F", attn="causal", pool="sep", method="ia3", epochs=20, final="retrain_all"),
    "f4_B_kd":             dict(model="B", teacher="b5_A1_causal_seppool", final="retrain_all"),   # control: b4_B_encoder
    "f5_A0_seppool_rslora": dict(model="A0", attn="causal", pool="sep", method="rslora", epochs=20, final="retrain_all"),
    "f6_A1_seppool_ctx96": dict(model="A1", attn="causal", pool="sep", win_len=1024, chunk_sents=96, batch=1),  # more context
})
QUEUES = {"final_acc1": ["f4_B_kd", "f2_A1_seppool_lora", "f6_A1_seppool_ctx96"],   # needs the Stage-B output (KD teacher)
          "final_acc2": ["f1_E_lorma", "f3_F_seppool_ia3", "f5_A0_seppool_rslora"],
          "stageA": [k for k in EXPERIMENTS if k.startswith("a")],
          "stageB": [k for k in EXPERIMENTS if k.startswith("b")],
          "stageC": [k for k in EXPERIMENTS if k.startswith("c")]}
RUN_QUEUE = QUEUES["final_acc1"]          # <- what this commit runs
MIN_HOURS_PER_EXP = 0.6

if P3_SMOKE:
    P1_SOURCES = {k: dict(dataset="", run_name=n) for k, n in {
        "A0": "SMOKE_A0_gpt2_110M_mix50_300M", "A1": "SMOKE_A1_modern_110M_mix50_300M",
        "B": "SMOKE_B_encdec_6x6_mix50_epoch_300M", "E": "SMOKE_E_prefixlm_110M_mix50_epoch_300M",
        "F": "SMOKE_F_localglobal_gqa_alibi_mix50_epoch_300M"}.items()}
    DEFAULTS.update(epochs=2, win_len=128, chunk_sents=16, sent_lstm=32)
    for k in EXPERIMENTS: EXPERIMENTS[k]["epochs"] = 2
    RUN_QUEUE = [x for x in os.environ.get("CS781_P3_QUEUE", "a3_E_window_lstm_pos").split(",") if x]

def exp_cfg(name):
    cfg = dict(DEFAULTS); cfg.update(EXPERIMENTS[name]); cfg["name"] = name
    if cfg["lr"] is None: cfg["lr"] = METHOD_LR[cfg["method"]]
    cfg["domains"] = tuple(cfg["domains"])
    return cfg

EXPS_DIR = ROOT / "p3" / "exps"; EXPS_DIR.mkdir(parents=True, exist_ok=True)
NUM_GPUS = torch.cuda.device_count()
print("GPUs:", NUM_GPUS, "| queue:", RUN_QUEUE)

GPUs: 2 | queue: ['f4_B_kd', 'f2_A1_seppool_lora', 'f6_A1_seppool_ctx96']


## 2. Inputs

In [3]:
DOCS_PATH = find_artifact("p3/data/docs.pkl")
assert DOCS_PATH is not None, "attach the P3_01 output (p3/data/docs.pkl)"
with open(DOCS_PATH, "rb") as f: DOCS = pickle.load(f)
LABELS = load_json(find_artifact("p3/data/labels.json")); NL = len(LABELS)
SEP_ID, PAD_ID = 0, 1
for d in DOCS: d["n"] = len(d["sent_ids"])
TRAIN_DOCS = [d for d in DOCS if d["split"] == "train"]
DEV_DOCS   = [d for d in DOCS if d["split"] == "dev"]
TEST_DOCS  = [d for d in DOCS if d["split"] == "test"]
print(f"docs: train {len(TRAIN_DOCS)} | dev {len(DEV_DOCS)} | test {len(TEST_DOCS)} | "
      f"sentences: {sum(d['n'] for d in DOCS)}")
DEV_PRESENT = sorted({l for d in DEV_DOCS for l in d["labels"]}, key=LABELS.index)
print("labels present in dev:", len(DEV_PRESENT), "| ceiling of the official dev macro-F1:", round(len(DEV_PRESENT) / NL, 4))

if IN_KAGGLE:   # results (small files) of earlier versions of this notebook
    n = 0
    for cand in glob.glob(f"/kaggle/input/**/{PROJECT}/p3/exps/*", recursive=True):
        for src in Path(cand).glob("*"):
            if src.suffix in (".json", ".zip", ".txt"):
                dst = EXPS_DIR / Path(cand).name / src.name
                if not dst.exists(): dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
    print("restored", n, "result files from attached earlier runs")

def find_p1_dir(key):
    src = P1_SOURCES[key]
    roots = [os.environ["CS781_P1_RUNS"]] if os.environ.get("CS781_P1_RUNS") else []
    roots += [str(ROOT / "runs")] + (["/kaggle/input"] if IN_KAGGLE else [])
    cands = sorted({c for r in roots for c in glob.glob(f"{r}/**/weights_final.pt", recursive=True)})
    by_run = [c for c in cands if Path(c).parent.name == src["run_name"]]
    by_ds = [c for c in cands if src["dataset"] and src["dataset"].lower() in [p.lower() for p in Path(c).parts]]
    hit = (by_run or by_ds or [None])[0]
    return Path(hit).parent if hit else None
P1_DIRS = {k: find_p1_dir(k) for k in P1_SOURCES}
for k, p in P1_DIRS.items(): print(f"  {k}: {p if p else 'NOT FOUND — attach its dataset or fix P1_SOURCES'}")
missing = [k for k in {exp_cfg(n)["model"] for n in RUN_QUEUE} if P1_DIRS[k] is None]
assert not missing, f"Phase-1 weights missing for {missing}"

docs: train 80 | dev 10 | test 10 | sentences: 21184
labels present in dev: 11 | ceiling of the official dev macro-F1: 0.8462
restored 30 result files from attached earlier runs
  A0: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a0-sc80/cs781_p1/runs/A0_gpt2_110M_SC80_epoch_1000M
  A1: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a1-sc80/cs781_p1/runs/A1_modern_110M_SC80_epoch_1000M
  B: /kaggle/input/datasets/aaravoswal/cs781-p1-model-b/cs781_p1/runs/B_encdec_6x6_SC80_epoch_1000M
  E: /kaggle/input/datasets/chrishitha2403/data-partial-e-prefixlm-110m-sc80-epoch-1000m/cs781_p1/runs/E_prefixlm_110M_SC80_epoch_1000M
  F: /kaggle/input/datasets/chrishitha2403/cs781-p1-model-f/cs781_p1/runs/F_localglobal_gqa_alibi_SC80_epoch_1000M


## 3. Model classes (unchanged from NB3)

In [4]:
# (model classes copied unchanged from NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

# ------------------------------------------------------------------ E: prefix-LM (encoder/hybrid adapted for LM)
class PrefixLM(ModernLM):
    """E: the A1 stack with a different ATTENTION MASK. Prompt (prefix) tokens attend to the whole prompt in both
    directions, like a BERT encoder; continuation tokens attend causally, like GPT. One shared stack plays both roles.
    Training: split each window at a random E, loss only on the continuation (predictions from position E−1 on).
    Generation: prefill the prompt with the bidirectional mask, then ordinary cached causal decoding."""
    def __init__(self, *a, enc_len_eval=768, **kw):
        super().__init__(*a, **kw); self.enc_len_eval = enc_len_eval
    def forward(self, idx, targets=None, enc_len=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        if targets is None:                                       # generation path (mask supplied by the runner)
            return super().forward(idx, None, pos, mask, cache, use_cache, last_only)
        T = idx.size(1); E = int(enc_len or min(self.enc_len_eval, T - 1))
        i = torch.arange(T, device=idx.device)
        allowed = (i[None, :] <= i[:, None]) | ((i[:, None] < E) & (i[None, :] < E))   # causal ∪ (prefix × prefix)
        logits, _, _ = super().forward(idx, None, None, allowed[None, None], None, False, False)
        loss = F.cross_entropy(logits[:, E - 1:].float().reshape(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return PrefixRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ F: efficient local-global decoder
def alibi_slopes(n):
    """ALiBi head slopes (Press et al. 2022): geometric sequence; extra heads interleaved when n is not a power of 2"""
    def pow2(m): return [2 ** (-8 * (k + 1) / m) for k in range(m)]
    if math.log2(n).is_integer(): return pow2(n)
    p = 2 ** math.floor(math.log2(n))
    return pow2(p) + pow2(2 * p)[0::2][: n - p]

class LocalGlobalAttention(nn.Module):
    """Grouped-query attention (h query heads share kv key/value heads) with an ALiBi distance penalty added to the
    attention logits (no position embeddings at all). window=W makes it sliding-window attention: a query only sees
    the previous W tokens; window=None is full causal attention."""
    def __init__(self, d, h, kv, window):
        super().__init__()
        self.h, self.kv, self.hd, self.window = h, kv, d // h, window
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, bias, past=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class EffBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, window):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.mlp = LocalGlobalAttention(d, h, kv, window), SwiGLU(d, hidden, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class EfficientLM(nn.Module):
    """F: lightweight decoder. Sliding-window attention (window W) in most layers, full causal attention in every
    k-th layer, grouped-query attention (fewer key/value heads → smaller KV-cache), ALiBi instead of position
    embeddings. Local layers model nearby text; the few global layers can still reach any earlier token."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=4, ffn_hidden=2304,
                 window=256, global_every=4, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        self.windows = [None if global_every and (i + 1) % global_every == 0 else window for i in range(n_layer)]
        self.blocks = nn.ModuleList([EffBlock(d_model, n_head, n_kv_head, ffn_hidden, dropout, w) for w in self.windows])
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head)), persistent=False)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _bias(self, qpos, kpos, key_ok, window):
        """(B|1, H, Tq, Tk) additive mask: −slope·distance for allowed pairs, −inf elsewhere"""
        dist = (qpos[:, :, None] - kpos[:, None, :]).float()               # (B|1, Tq, Tk)
        ok = (dist >= 0) & key_ok[:, None, :]
        if window is not None: ok = ok & (dist < window)
        b = -self.slopes.view(1, -1, 1, 1) * dist[:, None]
        return b.masked_fill(~ok[:, None], float("-inf"))
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if mask is None:                                          # training / plain forward: positions 0..T-1
            qpos = kpos = torch.arange(T, device=idx.device)[None]
            key_ok = torch.ones((1, T), dtype=torch.bool, device=idx.device)
        else:                                                     # generation: derive key positions from the mask
            key_ok = mask[:, 0, -1, :]                            # validity of every key (past + current)
            kpos = key_ok.long().cumsum(-1) - 1
            qpos = pos
        biases = {w: self._bias(qpos, kpos, key_ok, w) for w in set(self.windows)}
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, biases[self.windows[i]], cache[i] if cache is not None else None, use_cache); new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM,
                  "prefixlm": PrefixLM, "efficient": EfficientLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

## 4. Backbone: a Phase-1 model as a token encoder (+ optional soft prompt)
Same as Phase 2: LM head dropped; `attn="causal"` keeps the pretraining mask, `"bidir"` lets every token see the
whole window (E: exactly its pretrained prefix mode; B: its bidirectional encoder). A `prefix` (prompt tuning) is a
matrix of trainable vectors placed in front of the token embeddings of every window; its outputs are dropped.

In [5]:
class Backbone(nn.Module):
    def __init__(self, lm, arch, attn):
        super().__init__()
        self.lm, self.arch = lm, arch
        self.bidir = attn == "bidir" or arch == "encdec"
        self.d = lm.wte.weight.shape[1]
        if arch == "encdec": lm.dec = nn.ModuleList()
        self.ctx_limit = lm.ctx_len if arch == "gpt2" else (2 * lm.ctx_len if arch in ("modern", "prefixlm", "encdec") else 10 ** 9)
    def blocks(self):
        return self.lm.enc if self.arch == "encdec" else self.lm.blocks
    def forward(self, idx, valid, prefix=None):
        lm, (B, T), dev = self.lm, idx.shape, idx.device
        P = 0 if prefix is None else prefix.size(0)
        x = lm.wte(idx)
        if P:
            x = torch.cat([prefix[None].expand(B, -1, -1).to(x.dtype), x], 1)
            valid = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), valid], 1)
        L = T + P
        assert L <= self.ctx_limit, f"window {L} longer than the model's position range {self.ctx_limit}"
        pos = torch.arange(L, device=dev)[None]
        mask = valid[:, None, None, :] if self.bidir else None
        if self.arch == "encdec":
            cos, sin = lm._rope(0, L, dev)
            for blk in lm.enc: x = blk(x, cos, sin, mask)
            h = lm.enc_norm(x)
        elif self.arch == "gpt2":
            x = lm.drop(x + lm.wpe(pos))
            for blk in lm.blocks: x, _ = blk(x, None, mask, False)
            h = lm.ln_f(x)
        elif self.arch in ("modern", "prefixlm"):
            if lm.wpe is not None: x = x + lm.wpe(pos)
            cos, sin = lm.rope_cos[pos][:, None], lm.rope_sin[pos][:, None]
            x = lm.drop(x)
            for blk in lm.blocks: x, _ = blk(x, cos, sin, None, mask, False)
            h = lm.norm_f(x)
        elif self.arch == "efficient":
            dist = (pos[:, :, None] - pos[:, None, :]).float()
            biases = {}
            for w in set(lm.windows):
                if self.bidir:
                    ok = valid[:, None, :] & ((dist.abs() < w) if w is not None else torch.ones_like(dist, dtype=torch.bool))
                    ok = ok | torch.eye(L, dtype=torch.bool, device=dev)[None]
                    b = -lm.slopes.view(1, -1, 1, 1) * dist.abs()[:, None]
                else:
                    ok = dist >= 0
                    if w is not None: ok = ok & (dist < w)
                    b = -lm.slopes.view(1, -1, 1, 1) * dist[:, None]
                biases[w] = b.masked_fill(~ok[:, None], float("-inf"))
            for i, blk in enumerate(lm.blocks): x, _ = blk(x, biases[lm.windows[i]], None, False)
            h = lm.norm_f(x)
        else:
            raise ValueError(self.arch)
        return h[:, P:]

## 5. Sentence model
```
window tokens ─► backbone ─► token vectors ─► pool per sentence (mean of its tokens, or its end separator)
  ─► + position-in-document embedding + domain embedding ─► BiLSTM over the sentences of the chunk ─► linear (13)
                                                                                              └► or CRF over roles
```
* **Pooling**: mean of the sentence's token vectors (including its separator). `pool="sep"`: the vector at the
  separator after the sentence — for a causal model this is the only position that has seen the whole sentence.
* **Position**: roles sit in typical parts of a judgment (P3_01: facts early, ruling at the end); a learned
  embedding of the relative position (32 buckets) is added.
* **BiLSTM over sentences**: labels come in long blocks (87 % of sentences repeat the previous label) — reading the
  sequence of sentence vectors in both directions lets a sentence borrow evidence from its neighbours, and gives
  causal backbones right-hand context.
* **CRF** (optional): learned transition scores between roles (e.g. Precedent ↔ Ratio), Viterbi decoding.

In [6]:
class CRF(nn.Module):
    """linear-chain CRF over the 13 roles (no hard constraints); nll per sentence (same algorithm as Phase 2)"""
    def __init__(self, n):
        super().__init__()
        self.trans, self.start, self.end = nn.Parameter(torch.zeros(n, n)), nn.Parameter(torch.zeros(n)), nn.Parameter(torch.zeros(n))
    def full(self): return self.trans, self.start, self.end
    def nll(self, em, tags, mask):
        tr, st, en = self.full(); tags = tags.clamp(min=0)
        score = st[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        alpha = st[None] + em[:, 0]
        for t in range(1, em.size(1)):
            m = mask[:, t]
            score = score + (tr[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)) * m
            nxt = torch.logsumexp(alpha[:, :, None] + tr[None] + em[:, t, None, :], dim=1)
            alpha = torch.where(m[:, None], nxt, alpha)
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        return (torch.logsumexp(alpha + en[None], 1) - score - en[last]).sum() / mask.sum().clamp(min=1)

NPOS = 32
class SentenceHead(nn.Module):
    def __init__(self, d, cfg):
        super().__init__()
        self.drop = nn.Dropout(cfg["head_dropout"])
        self.pos = nn.Embedding(NPOS, d) if cfg["pos_emb"] else None
        self.dom = nn.Embedding(2, d) if cfg["domain_emb"] else None
        for e in (self.pos, self.dom):
            if e is not None: nn.init.normal_(e.weight, 0, 0.02)
        h = cfg["sent_lstm"]
        self.lstm = nn.LSTM(d, h, batch_first=True, bidirectional=True) if h else None
        self.proj = nn.Linear(2 * h if h else d, NL)
        self.crf = CRF(NL) if cfg["head"] == "crf" else None
    def forward(self, s, smask, posb, dom):
        """s (B, S, d) sentence vectors, smask (B, S) bool, posb (B, S) long, dom (B,) long -> (B, S, 13) scores"""
        with torch.autocast(device_type=s.device.type, enabled=False):
            x = s.float()
            if self.pos is not None: x = x + self.pos(posb)
            if self.dom is not None: x = x + self.dom(dom)[:, None]
            x = self.drop(x)
            if self.lstm is not None:
                lengths = smask.sum(1).clamp(min=1).cpu()
                packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
                out, _ = self.lstm(packed)
                x, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
                x = self.drop(x)
            return self.proj(x)

class RRModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone, self.pool = backbone, cfg["pool"]
        self.prompt = None
        if cfg["method"] == "prompt":     # initialised from random token embeddings (Lester et al. 2021)
            emb = backbone.lm.wte.weight.detach()
            lo = min(1000, emb.size(0) // 2)                  # skip special / very rare-id tokens
            self.prompt = nn.Parameter(emb[torch.randint(lo, emb.size(0), (cfg["prompt_len"],))].clone())
        self.head = SentenceHead(backbone.d, cfg)
    def sentence_vectors(self, idx, valid, seg, n_sent):
        """seg (W, T): index of the sentence (in this batch) each token belongs to, -1 for padding;
        returns (n_sent, d): mean of the sentence's tokens, or the vector at its last token (= separator)"""
        h = self.backbone(idx, valid, self.prompt)
        d = h.size(-1); flat_h = h.reshape(-1, d); flat_s = seg.reshape(-1); keep = flat_s >= 0
        hs, ss = flat_h[keep].float(), flat_s[keep]
        if self.pool == "sep":
            pos = torch.arange(flat_s.numel(), device=idx.device)[keep]
            last = torch.full((n_sent,), -1, dtype=torch.long, device=idx.device).scatter_reduce(0, ss, pos, "amax")
            return flat_h[last].float()
        tot = torch.zeros(n_sent, d, device=idx.device).index_add_(0, ss, hs)
        cnt = torch.zeros(n_sent, device=idx.device).index_add_(0, ss, torch.ones_like(ss, dtype=torch.float))
        return tot / cnt.clamp(min=1)[:, None]
    def forward(self, batch):
        sv = self.sentence_vectors(batch["idx"], batch["valid"], batch["seg"], batch["n_sent"])
        s = sv[batch["sent_index"]] * batch["smask"][..., None]                  # (B, S, d)
        return self.head(s, batch["smask"], batch["posb"], batch["dom"])

## 6. Adaptation techniques (all written here)

| method | change to a frozen linear layer `y = W0 x` | trainable |
|---|---|---|
| `full` | everything trains | 100 % |
| `frozen` | nothing (head only) | head |
| `lora` (Hu et al. 2021) | `y + (α/r)·B A x`, B = 0 at start | ~2.5 % |
| `rslora` (Kalajdzievski 2023) | `y + (α/√r)·B A x` — "rank-stabilised" scale, so the update's size does not shrink as r grows | ~2.5 % |
| `lorma` + (Bihany et al. 2025) | `(I + (α/r)·B A)·y` — the low-rank matrix **multiplies** the frozen output | ~2.7 % |
| `lorma` π | `I_π(B A)·y`, permutation rank inflation, identity at start | ~1 % |
| `ia3` (Liu et al. 2022) | element-wise **rescaling**: keys and values `k ⊙ l_k`, `v ⊙ l_v` and the FFN hidden activation `h ⊙ l_ff`; l = 1 at start | ~0.06 % |
| `prompt` (Lester et al. 2021) | backbone fully frozen; `prompt_len` trainable input vectors are prepended to every window | ~0.01 % |
| `bitfit`, `adapter` | as in Phase 2 | |

The sentence-level head (position/domain embeddings, BiLSTM, classifier) is trained in every method.

In [7]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.scale = base, alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

_PERM_IDX, _PERM_LOCK = {}, threading.Lock()
def perm_index(d, device):
    key = (d, str(device))
    with _PERM_LOCK:
        if key not in _PERM_IDX:
            i = torch.arange(d, device=device); _PERM_IDX[key] = (i[None, :] - i[:, None]) % d
        return _PERM_IDX[key]

class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        d = base.out_features; self.base, self.variant = base, variant
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r))
        else:
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.empty(d, r); nn.init.kaiming_uniform_(B, a=math.sqrt(5)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = self.B @ self.A
        return F.linear(y, M.gather(1, perm_index(M.size(0), M.device)))

class IA3Linear(nn.Module):
    """(IA)^3: mode "out" rescales the layer's outputs (keys/values; `mask` limits it to part of a fused q|k|v
    output), mode "in" rescales its inputs (= the FFN hidden activation when wrapped around the down-projection)"""
    def __init__(self, base, mode, mask=None):
        super().__init__()
        self.base, self.mode = base, mode
        n = base.out_features if mode == "out" else base.in_features
        self.l = nn.Parameter(torch.ones(n))
        self.register_buffer("mask", mask if mask is not None else torch.ones(n, dtype=torch.bool), persistent=False)
    def forward(self, x):
        if self.mode == "in": return self.base(x * self.l.to(x.dtype))
        s = torch.where(self.mask, self.l, torch.ones_like(self.l))
        return self.base(x) * s.to(x.dtype)

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class AdapterLinear(nn.Module):
    def __init__(self, base, dim):
        super().__init__()
        d = base.out_features; self.base = base
        self.down, self.up = nn.Linear(d, dim), nn.Linear(dim, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x); return y + self.up(F.gelu(self.down(y)))

ATTN_TYPES = (CausalSelfAttention, ModernAttention, LocalGlobalAttention)
MLP_TYPES = (MLP, SwiGLU, GeluMLP)

def target_linears(backbone, which):
    out = []
    for parent in list(backbone.blocks().modules()):
        kind = "attn" if isinstance(parent, ATTN_TYPES) else "mlp" if isinstance(parent, MLP_TYPES) else None
        if kind is None or (which == "attn" and kind != "attn"): continue
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear):
                out.append((parent, name, child, bool(getattr(child, "RESIDUAL_SCALE", False))))
    return out

def apply_method(backbone, cfg):
    m = cfg["method"]
    for p in backbone.parameters(): p.requires_grad = (m == "full")
    if m in ("full", "frozen", "prompt"): return
    if m == "topk":
        for blk in list(backbone.blocks())[-cfg["topk"]:]:
            for p in blk.parameters(): p.requires_grad = True
        return
    if m == "ia3":
        for parent in list(backbone.blocks().modules()):
            if isinstance(parent, CausalSelfAttention):                      # fused q|k|v: rescale only k and v
                d = parent.qkv.in_features
                parent.qkv = IA3Linear(parent.qkv, "out", torch.arange(3 * d) >= d)
            elif isinstance(parent, (ModernAttention, LocalGlobalAttention)):
                parent.kvp = IA3Linear(parent.kvp, "out")
            elif isinstance(parent, SwiGLU): parent.w2 = IA3Linear(parent.w2, "in")
            elif isinstance(parent, (MLP, GeluMLP)): parent.proj = IA3Linear(parent.proj, "in")
        return
    which = cfg["lora_targets"] if m in ("lora", "rslora", "lorma") else "all"
    for parent, name, lin, is_out in target_linears(backbone, which):
        if m in ("lora", "rslora"): setattr(parent, name, LoRALinear(lin, cfg["lora_r"], cfg["lora_alpha"], rs=(m == "rslora")))
        elif m == "lorma": setattr(parent, name, LoRMALinear(lin, cfg["lora_r"], cfg["lora_alpha"], cfg["lorma_variant"]))
        elif m == "adapter":
            if is_out: setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        elif m == "bitfit":
            if lin.bias is not None: lin.bias.requires_grad = True
            else: setattr(parent, name, BiasLinear(lin))
    if m == "bitfit":
        for mod in backbone.modules():
            if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                for p in mod.parameters(): p.requires_grad = True

def set_backbone_dropout(backbone, p):
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = p

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg, f"no run_info.json/summary.json next to {d}"
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

def build_model(cfg, device):
    pcfg, sd = load_phase1(cfg["model"])
    vocab = sd["wte.weight"].shape[0]
    torch.manual_seed(cfg["seed"])
    lm = MODEL_REGISTRY[pcfg["arch"]](vocab=vocab, **{k: v for k, v in pcfg.items() if k != "vocab"})
    if cfg["init"] == "pretrained": lm.load_state_dict(sd)
    backbone = Backbone(lm, pcfg["arch"], cfg["attn"])
    set_backbone_dropout(backbone, cfg["backbone_dropout"])
    apply_method(backbone, cfg)
    model = RRModel(backbone, cfg).to(device)
    return model, pcfg

def count_parameters(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen: continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

## 7. Chunks and windows
* A **chunk** = up to `chunk_sents` consecutive sentences of one document (training: random start offset each epoch;
  inference: chunks overlap by half and each sentence's scores are averaged, weighted towards the chunk centre).
* Inside a chunk, sentences (each ≤ `max_sent_tok` tokens + one separator token) are **packed whole** into windows
  of ≤ `win_len` tokens (`ctx="window"`), or each sentence gets its own window (`ctx="sentence"`).
* Every token carries the index of its sentence → mean-pooling with `index_add`.

In [8]:
def window_capacity(cfg):
    return cfg["win_len"] - (cfg["prompt_len"] if cfg["method"] == "prompt" else 0)

def sent_tokens(doc, i, cfg):
    ids = doc["sent_ids"][i][:min(cfg["max_sent_tok"], window_capacity(cfg) - 1)].astype(np.int64)
    return np.concatenate([ids, [SEP_ID]])

def make_chunk(doc, start, cfg):
    n = doc["n"]; end = min(n, start + cfg["chunk_sents"])
    cap = window_capacity(cfg)
    windows, cur, cur_len = [], [], 0
    for i in range(start, end):
        t = sent_tokens(doc, i, cfg)
        if cfg["ctx"] == "sentence" or (cur and cur_len + len(t) > cap):
            if cur: windows.append(cur)
            cur, cur_len = [], 0
        cur.append((i - start, t)); cur_len += len(t)
    if cur: windows.append(cur)
    posb = np.minimum((np.arange(start, end) * NPOS) // max(1, n), NPOS - 1)
    ch = {"doc": doc["id"], "start": start, "len": end - start, "windows": windows, "posb": posb,
          "dom": 0 if doc["domain"] == "CL" else 1}
    if doc.get("label_ids") is not None: ch["labels"] = doc["label_ids"][start:end]
    return ch

def chunk_starts(n, cfg, train, rng=None):
    S = cfg["chunk_sents"]
    if n <= S: return [0]
    stride = S if train else max(1, S // 2)
    off = int(rng.integers(0, stride)) if (train and rng is not None) else 0
    return sorted({0, n - S, *range(off, n - S, stride)})

def collate(chunks, device):
    rows, seg_rows, sent_index, n_sent = [], [], [], 0
    Smax = max(c["len"] for c in chunks)
    sent_index = np.zeros((len(chunks), Smax), np.int64); smask = np.zeros((len(chunks), Smax), bool)
    posb = np.zeros((len(chunks), Smax), np.int64); labels = np.full((len(chunks), Smax), -100, np.int64)
    for b, c in enumerate(chunks):
        base = n_sent
        for w in c["windows"]:
            rows.append(np.concatenate([t for _, t in w]))
            seg_rows.append(np.concatenate([np.full(len(t), base + j) for j, t in w]))
        sent_index[b, :c["len"]] = base + np.arange(c["len"]); smask[b, :c["len"]] = True
        posb[b, :c["len"]] = c["posb"]
        if "labels" in c: labels[b, :c["len"]] = c["labels"]
        n_sent += c["len"]
    T = max(len(r) for r in rows)
    idx = np.full((len(rows), T), PAD_ID, np.int64); valid = np.zeros((len(rows), T), bool); seg = np.full((len(rows), T), -1, np.int64)
    for k, (r, s) in enumerate(zip(rows, seg_rows)):
        idx[k, :len(r)] = r; valid[k, :len(r)] = True; seg[k, :len(s)] = s
    t = lambda a: torch.from_numpy(a).to(device, non_blocking=True)
    return {"idx": t(idx), "valid": t(valid), "seg": t(seg), "n_sent": n_sent, "sent_index": t(sent_index),
            "smask": t(smask), "posb": t(posb), "dom": t(np.array([c["dom"] for c in chunks], np.int64)), "labels": t(labels)}

LABEL_COUNTS = np.bincount(np.concatenate([d["label_ids"] for d in TRAIN_DOCS]), minlength=NL).astype(float)
def class_weights(kind):
    if kind is None: return None
    w = np.sqrt(LABEL_COUNTS.max() / np.maximum(LABEL_COUNTS, 1)); w = np.minimum(w, 10.0)
    return torch.tensor(w / w.mean(), dtype=torch.float32)
print("train label counts:", dict(zip(LABELS, LABEL_COUNTS.astype(int))))
print("class weights (sqrt_inv):", dict(zip(LABELS, class_weights("sqrt_inv").numpy().round(2))))

train label counts: {'Fact': np.int64(4098), 'Issue': np.int64(107), 'ArgumentPetitioner': np.int64(1613), 'ArgumentRespondent': np.int64(1348), 'Statute': np.int64(1099), 'Dissent': np.int64(284), 'PrecedentReliedUpon': np.int64(1945), 'PrecedentNotReliedUpon': np.int64(569), 'PrecedentOverruled': np.int64(3), 'RulingByLowerCourt': np.int64(874), 'RatioOfTheDecision': np.int64(4301), 'RulingByPresentCourt': np.int64(307), 'None': np.int64(297)}
class weights (sqrt_inv): {'Fact': np.float32(0.32), 'Issue': np.float32(1.98), 'ArgumentPetitioner': np.float32(0.51), 'ArgumentRespondent': np.float32(0.56), 'Statute': np.float32(0.62), 'Dissent': np.float32(1.21), 'PrecedentReliedUpon': np.float32(0.46), 'PrecedentNotReliedUpon': np.float32(0.86), 'PrecedentOverruled': np.float32(3.12), 'RulingByLowerCourt': np.float32(0.69), 'RatioOfTheDecision': np.float32(0.31), 'RulingByPresentCourt': np.float32(1.17), 'None': np.float32(1.19)}


## 8. Inference, decoding, metric, dev tuning, error analysis
* Per document: scores of every sentence averaged over the (overlapping) chunks containing it.
* Decoding: argmax, or Viterbi with the CRF's transitions.
* **Official metric** (`evaluate_phase3.py`): sentences of all documents pooled, macro-F1 over **all 13 labels**
  (absent labels count as 0). We also report the macro-F1 over the labels present in dev and per domain.
* Dev tuning: per-label decision biases (coordinate search, kept only if dev improves by ≥ 0.005).

In [9]:
@torch.no_grad()
def doc_scores(model, docs, cfg, device, max_chunks=8):
    model.eval(); use_amp = device.type == "cuda"
    acc = {d["id"]: np.zeros((d["n"], NL), np.float32) for d in docs}; ws = {d["id"]: np.zeros(d["n"], np.float32) for d in docs}
    chunks = [make_chunk(d, s, cfg) for d in docs for s in chunk_starts(d["n"], cfg, train=False)]
    for i in range(0, len(chunks), max_chunks):
        part = chunks[i:i + max_chunks]; batch = collate(part, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            sc = model(batch)
        sc = sc.float()
        if model.head.crf is None: sc = F.log_softmax(sc, -1)
        sc = sc.cpu().numpy()
        for b, c in enumerate(part):
            n = c["len"]; j = np.arange(n); w = np.minimum(j + 1, n - j).astype(np.float32)
            acc[c["doc"]][c["start"]:c["start"] + n] += sc[b, :n] * w[:, None]
            ws[c["doc"]][c["start"]:c["start"] + n] += w
    return {k: acc[k] / np.maximum(ws[k], 1e-6)[:, None] for k in acc}

def viterbi(em, trans, start, end):
    n = len(em); score = start + em[0]; bp = np.zeros((n, trans.shape[0]), np.int64)
    for t in range(1, n):
        cand = score[:, None] + trans; bp[t] = cand.argmax(0); score = cand.max(0) + em[t]
    path = np.zeros(n, np.int64); path[-1] = (score + end).argmax()
    for t in range(n - 1, 0, -1): path[t - 1] = bp[t, path[t]]
    return path

def crf_numpy(model):
    if model.head.crf is None: return None
    return tuple(x.detach().float().cpu().numpy() for x in model.head.crf.full())

def decode(scores, docs, crf_params=None, bias=None):
    bv = np.zeros(NL, np.float32)
    for l, v in (bias or {}).items(): bv[LABELS.index(l)] += v
    out = {}
    for d in docs:
        s = scores[d["id"]] + bv[None]
        ids = viterbi(s, *crf_params) if crf_params is not None else s.argmax(1)
        out[d["id"]] = [LABELS[i] for i in ids]
    return out

def rr_scores(truth, pred, labels=None):
    """identical arithmetic to evaluate_phase3.py (labels=None -> all 13)"""
    labels = labels or LABELS; yt, yp = [], []
    for did in truth:
        assert len(truth[did]) == len(pred[did]), f"{did}: sentence-count mismatch"
        yt += list(truth[did]); yp += list(pred[did])
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=LABELS, zero_division=0)
    return {"macro_f1": float(f1_score(yt, yp, labels=labels, average="macro", zero_division=0)),
            "macro_p": float(precision_score(yt, yp, labels=labels, average="macro", zero_division=0)),
            "macro_r": float(recall_score(yt, yp, labels=labels, average="macro", zero_division=0)),
            "accuracy": float(np.mean(np.array(yt) == np.array(yp))) if yt else 0.0,
            "per_label": {l: {"P": float(p[i]), "R": float(r[i]), "F1": float(f[i]), "support": int(s[i])} for i, l in enumerate(LABELS)}}

def tune_bias(scores, docs, truth, crf_params, log):
    base = rr_scores(truth, decode(scores, docs, crf_params))["macro_f1"]; cur, best = {}, base
    for l in LABELS:
        for v in (-2.0, -1.0, -0.5, 0.5, 1.0, 2.0):
            s = rr_scores(truth, decode(scores, docs, crf_params, {**cur, l: v}))["macro_f1"]
            if s > best: best, cur = s, {**cur, l: v}
    keep = cur if best >= base + 0.005 else {}
    log(f"bias tuning on dev: {base:.4f} -> {best:.4f} {'(kept)' if keep else '(not kept)'} {keep}")
    return keep, base, best

def error_report(truth, pred, docs):
    yt = [l for k in truth for l in truth[k]]; yp = [l for k in truth for l in pred[k]]
    cm = confusion_matrix(yt, yp, labels=LABELS)
    conf = Counter({(LABELS[i], LABELS[j]): int(cm[i, j]) for i in range(NL) for j in range(NL) if i != j and cm[i, j]})
    per_dom = {}
    for dom in ("CL", "IT"):
        ids = [d["id"] for d in docs if d["domain"] == dom and d["id"] in truth]
        if ids:
            sc = rr_scores({k: truth[k] for k in ids}, {k: pred[k] for k in ids})
            per_dom[dom] = {"macro_f1": sc["macro_f1"], "accuracy": sc["accuracy"],
                            "macro_f1_present": rr_scores({k: truth[k] for k in ids}, {k: pred[k] for k in ids},
                                                          sorted({l for k in ids for l in truth[k]}, key=LABELS.index))["macro_f1"]}
    return {"confusions(gold->pred)": {f"{a}->{b}": n for (a, b), n in conf.most_common(25)},
            "confusion_matrix": cm.tolist(), "per_domain": per_dom}

def write_predictions(pred, docs, path):
    out = {}
    for d in docs:
        p = pred[d["id"]]; assert len(p) == d["n"] and all(l in LABELS for l in p), d["id"]
        out[d["id"]] = list(p)
    save_json(out, path); return out

def make_zip(pred_json_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z: z.write(pred_json_path, "prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]; obj = json.loads(z.read("prediction.json"))
    assert set(obj) == {d["id"] for d in TEST_DOCS}, "submission ids != test ids"
    assert all(len(obj[d["id"]]) == d["n"] for d in TEST_DOCS), "sentence counts differ"

## 9. Training (+ knowledge distillation), experiment runner
AdamW, warmup + linear decay, fp16 autocast on GPU, gradient clipping; backbone and head have separate learning
rates. After every epoch the dev documents are decoded with the official metric; the best epoch is kept.

**Knowledge distillation** (`teacher=<finished experiment>`): the teacher's per-sentence probability distribution
over the 13 roles (merged over its own chunks) is the soft target:
`loss = α·CE(gold) + (1−α)·T²·KL(softmax(teacher/T) ‖ softmax(student/T))`.

In [10]:
def load_teacher(name, device):
    tcfg = dict(DEFAULTS); tcfg.update(load_json(EXPS_DIR / name / "config.json") or load_json(find_artifact(f"p3/exps/{name}/config.json")) or {})
    tcfg["domains"] = tuple(tcfg["domains"])
    w = find_artifact(f"p3/exps/{name}/trainable_devmodel.pt")
    assert w is not None, f"teacher weights p3/exps/{name}/trainable_devmodel.pt not found — attach the run that trained it"
    assert tcfg["head"] == "linear", "teacher must have a linear head"
    teacher, _ = build_model(tcfg, device)
    sd = torch.load(w, map_location="cpu", weights_only=False)
    res = teacher.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return teacher, tcfg

def teacher_targets(name, docs, device, log):
    teacher, tcfg = load_teacher(name, device)
    t0 = time.time(); out = doc_scores(teacher, docs, tcfg, device); tot, _ = count_parameters(teacher)
    log(f"teacher {name} ({tcfg['model']}, {tot/1e6:.1f}M params): soft targets for {len(docs)} docs in {time.time()-t0:.0f}s")
    del teacher
    if device.type == "cuda": torch.cuda.empty_cache()
    return out, tot

def train_model(cfg, train_docs, device, epochs, log, dev_docs=None):
    soft, teacher_params = None, None
    if cfg["teacher"]:
        assert cfg["head"] == "linear"; soft, teacher_params = teacher_targets(cfg["teacher"], train_docs, device, log)
    model, pcfg = build_model(cfg, device)
    total, trainable = count_parameters(model)
    rng = np.random.default_rng(cfg["seed"])
    hp = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head.")]
    bp = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    groups = [{"params": [p for p in hp if p.ndim >= 2], "lr": cfg["lr_head"], "weight_decay": cfg["weight_decay"]},
              {"params": [p for p in hp if p.ndim < 2], "lr": cfg["lr_head"], "weight_decay": 0.0}]
    if bp and cfg["lr"] > 0:
        groups += [{"params": [p for p in bp if p.ndim >= 2 and p is not model.prompt], "lr": cfg["lr"], "weight_decay": cfg["weight_decay"]},
                   {"params": [p for p in bp if p.ndim < 2 or p is model.prompt], "lr": cfg["lr"], "weight_decay": 0.0}]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
    for g in opt.param_groups: g["base_lr"] = g["lr"]
    n_chunks = sum(len(chunk_starts(d["n"], cfg, True, None)) for d in train_docs)
    total_steps = max(1, epochs * math.ceil(n_chunks / cfg["batch"])); warm = max(1, int(cfg["warmup"] * total_steps))
    use_amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw = class_weights(cfg["class_weight"]); cw = cw.to(device) if cw is not None else None
    tset = {n for n, p in model.named_parameters() if p.requires_grad}
    n_win = sum(len(make_chunk(d, s, cfg)["windows"]) for d in train_docs for s in chunk_starts(d["n"], cfg, True, None))
    log(f"model {cfg['model']} ({pcfg['arch']}) attn={cfg['attn']} method={cfg['method']} ctx={cfg['ctx']} lstm={cfg['sent_lstm']} "
        f"pos={cfg['pos_emb']} head={cfg['head']} domains={cfg['domains']} | params {total/1e6:.1f}M, trainable {trainable/1e6:.3f}M "
        f"({100*trainable/total:.2f}%) | {n_chunks} chunks ≈ {n_win} windows/epoch, {total_steps} steps")
    if use_amp: torch.cuda.reset_peak_memory_stats(device)
    dev_truth = {d["id"]: d["labels"] for d in (dev_docs or [])}
    hist, best, step, t0 = [], (-1.0, None, 0), 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        chunks = [make_chunk(d, s, cfg) for d in train_docs for s in chunk_starts(d["n"], cfg, True, rng)]
        order = rng.permutation(len(chunks)); run_loss, nb = 0.0, 0
        for i in range(0, len(order), cfg["batch"]):
            part = [chunks[j] for j in order[i:i + cfg["batch"]]]; batch = collate(part, device)
            lr_mult = (step + 1) / warm if step < warm else max(0.0, (total_steps - step) / max(1, total_steps - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_mult
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                sc = model(batch)
            sc = sc.float(); lab = batch["labels"]
            if model.head.crf is not None: loss = model.head.crf.nll(sc, lab, batch["smask"])
            else: loss = F.cross_entropy(sc.reshape(-1, NL), lab.reshape(-1), weight=cw, ignore_index=-100)
            if soft is not None:
                T, a = cfg["kd_T"], cfg["kd_alpha"]
                tt = np.zeros((len(part), sc.size(1), NL), np.float32)
                for b, c in enumerate(part): tt[b, :c["len"]] = soft[c["doc"]][c["start"]:c["start"] + c["len"]]
                t = torch.from_numpy(tt).to(device) / T
                kl = (F.softmax(t, -1) * (F.log_softmax(t, -1) - F.log_softmax(sc / T, -1))).sum(-1)
                m = batch["smask"].float(); loss = a * loss + (1 - a) * T * T * (kl * m).sum() / m.sum().clamp(min=1)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], cfg["grad_clip"])
            scaler.step(opt); scaler.update()
            run_loss += loss.item(); nb += 1; step += 1
        msg = {"epoch": ep, "loss": run_loss / max(nb, 1), "min": (time.time() - t0) / 60}
        if dev_docs:
            scs = doc_scores(model, dev_docs, cfg, device)
            msg["dev_f1"] = rr_scores(dev_truth, decode(scs, dev_docs, crf_numpy(model)))["macro_f1"]
            if msg["dev_f1"] > best[0]:
                best = (msg["dev_f1"], {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if n in tset}, ep)
        hist.append(msg); log(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in msg.items()}))
    info = {"total_params": total, "trainable_params": trainable, "train_min": (time.time() - t0) / 60,
            "peak_mem_gb": torch.cuda.max_memory_allocated(device) / 1e9 if use_amp else None, "history": hist,
            "backbone_arch": pcfg["arch"], "teacher_params": teacher_params, "windows_per_epoch": n_win}
    if dev_docs and best[1] is not None:
        model.load_state_dict(best[1], strict=False); info.update(best_epoch=best[2], best_dev_f1=best[0])
    return model, info, tset

def save_trainable(model, names, path):
    sd = {n: p.detach().half().cpu() if p.is_floating_point() else p.detach().cpu() for n, p in model.named_parameters() if n in names}
    torch.save(sd, str(path) + ".tmp"); os.replace(str(path) + ".tmp", path)

def run_experiment(name, device):
    cfg = exp_cfg(name); d = EXPS_DIR / name; d.mkdir(parents=True, exist_ok=True)
    if (d / "result.json").exists(): print(f"[{name}] already done — skipped"); return load_json(d / "result.json")
    logf = open(d / "log.txt", "a")
    def log(msg):
        line = f"[{name} | {device}] {msg}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    save_json(cfg, d / "config.json")
    doms = set(cfg["domains"])
    tr_docs = [x for x in TRAIN_DOCS if x["domain"] in doms]; dv_docs = [x for x in DEV_DOCS if x["domain"] in doms]
    te_docs = [x for x in TEST_DOCS if x["domain"] in doms]
    truth = {x["id"]: x["labels"] for x in dv_docs}
    model, info, names = train_model(cfg, tr_docs, device, cfg["epochs"], log, dev_docs=dv_docs)
    t0 = time.time(); sc_dev = doc_scores(model, dv_docs, cfg, device)
    info["infer_sec_per_doc"] = (time.time() - t0) / max(1, len(dv_docs))
    crfp = crf_numpy(model)
    raw = rr_scores(truth, decode(sc_dev, dv_docs, crfp))
    bias, _, _ = tune_bias(sc_dev, dv_docs, truth, crfp, log) if cfg["tune_bias"] else ({}, 0, 0)
    pred_dev = decode(sc_dev, dv_docs, crfp, bias); fin = rr_scores(truth, pred_dev)
    present = rr_scores(truth, pred_dev, sorted({l for v in truth.values() for l in v}, key=LABELS.index))
    er = error_report(truth, pred_dev, dv_docs)
    save_json({"raw": raw, "final": fin, "present_labels_macro_f1": present["macro_f1"], "bias": bias, **er}, d / "dev_metrics.json")
    write_predictions(pred_dev, dv_docs, d / "dev_prediction.json")
    save_trainable(model, names, d / "trainable_devmodel.pt")
    write_predictions(decode(doc_scores(model, te_docs, cfg, device), te_docs, crfp, bias), te_docs, d / "prediction_devmodel.json")
    final_src = "prediction_devmodel.json"
    if cfg["final"] == "retrain_all":
        del model
        if device.type == "cuda": torch.cuda.empty_cache()
        log(f"retraining on train+dev for {info.get('best_epoch', cfg['epochs'])} epochs")
        model, _, names = train_model(cfg, tr_docs + dv_docs, device, info.get("best_epoch", cfg["epochs"]), log)
        save_trainable(model, names, d / "trainable_all.pt")
        write_predictions(decode(doc_scores(model, te_docs, cfg, device), te_docs, crf_numpy(model), bias), te_docs, d / "prediction_all.json")
        final_src = "prediction_all.json"
    if doms == {"CL", "IT"}: make_zip(d / final_src, d / "submission.zip")
    row = {"name": name, "model": cfg["model"], "arch": info["backbone_arch"], "attn": cfg["attn"] if cfg["model"] != "B" else "bidir(enc)",
           "init": cfg["init"], "method": cfg["method"] + (f"-{cfg['lorma_variant']}" if cfg["method"] == "lorma" else "")
           + (f"+KD({cfg['teacher']})" if cfg["teacher"] else ""),
           "ctx": cfg["ctx"], "lstm": cfg["sent_lstm"], "pos": cfg["pos_emb"], "head": cfg["head"], "pool": cfg["pool"],
           "domains": "+".join(cfg["domains"]),
           "dev_macro_f1": fin["macro_f1"], "dev_f1_raw": raw["macro_f1"], "dev_f1_present": present["macro_f1"],
           "dev_macro_p": fin["macro_p"], "dev_macro_r": fin["macro_r"], "dev_acc": fin["accuracy"],
           **{f"{k}_f1": v["macro_f1"] for k, v in er["per_domain"].items()},
           **{f"F1_{l}": fin["per_label"][l]["F1"] for l in LABELS},
           "best_epoch": info.get("best_epoch"), "epochs": cfg["epochs"],
           "total_params_M": info["total_params"] / 1e6, "trainable_params_M": info["trainable_params"] / 1e6,
           "trainable_%": 100 * info["trainable_params"] / info["total_params"], "teacher_params_M": (info["teacher_params"] or 0) / 1e6,
           "peak_mem_GB": info["peak_mem_gb"], "train_min": info["train_min"], "infer_s_per_doc": info["infer_sec_per_doc"],
           "bias_tuned": bool(bias), "submission": f"p3/exps/{name}/submission.zip ({final_src})" if doms == {"CL", "IT"} else "(single domain)"}
    save_json({"row": row, "info": info, "cfg": cfg}, d / "result.json")
    log(f"DONE dev macro-F1 {fin['macro_f1']:.4f} (raw {raw['macro_f1']:.4f}, over present labels {present['macro_f1']:.4f}) "
        f"| per domain {json.dumps({k: round(v['macro_f1'], 4) for k, v in er['per_domain'].items()})}")
    logf.close(); del model
    if device.type == "cuda": torch.cuda.empty_cache()
    return row

## 10. Unit tests (CPU, seconds)
CRF = brute force; LoRA / rsLoRA / LoRMA± / IA³ / BitFit / adapters start as the identity; chunking covers every
sentence exactly; pooling is a correct per-sentence mean; the official metric copy gives 1.0 × (present/13) on gold.

In [11]:
def _tests():
    torch.manual_seed(0); n, T = 3, 4
    crf = CRF(n)
    with torch.no_grad(): crf.trans.normal_(); crf.start.normal_(); crf.end.normal_()
    em = torch.randn(1, T, n); tags = torch.tensor([[0, 2, 1, 1]]); mask = torch.ones(1, T, dtype=torch.bool)
    def sc(y):
        s = crf.start[y[0]] + em[0, 0, y[0]] + crf.end[y[-1]]
        for t in range(1, T): s = s + crf.trans[y[t - 1], y[t]] + em[0, t, y[t]]
        return s
    allp = list(itertools.product(range(n), repeat=T))
    logZ = torch.logsumexp(torch.stack([sc(y) for y in allp]), 0)
    assert abs(T * crf.nll(em, tags, mask).item() - (logZ - sc(tags[0].tolist())).item()) < 1e-4, "CRF"
    tr, st, en = (x.detach().numpy() for x in crf.full())
    assert tuple(viterbi(em[0].numpy(), tr, st, en).tolist()) == max(allp, key=lambda y: sc(y).item()), "Viterbi"
    base = nn.Linear(16, 24, bias=False); x = torch.randn(5, 16); y = base(x)
    mods = [LoRALinear(base, 4, 8), LoRALinear(base, 4, 8, rs=True), LoRMALinear(base, 4, 8, "plus"), LoRMALinear(base, 4, 8, "pi"),
            IA3Linear(base, "out"), IA3Linear(base, "in"), BiasLinear(base), AdapterLinear(base, 6)]
    for mod in mods: assert torch.allclose(mod(x), y, atol=1e-5), type(mod).__name__
    cfg = dict(DEFAULTS); cfg.update(method="full")
    for ctx in ("window", "sentence"):
        cfg["ctx"] = ctx
        for d in DOCS[:6]:
            for train in (True, False):
                cover = np.zeros(d["n"], int)
                for s in chunk_starts(d["n"], cfg, train, np.random.default_rng(1) if train else None):
                    ch = make_chunk(d, s, cfg)
                    got = [j for w in ch["windows"] for j, _ in w]
                    assert got == list(range(ch["len"])), "packing must keep every sentence once, in order"
                    assert all(sum(len(t) for _, t in w) <= cfg["win_len"] for w in ch["windows"])
                    cover[s:s + ch["len"]] += 1
                assert (cover >= 1).all(), "a sentence not covered by any chunk"
    # pooling = mean of each sentence's token vectors
    class _BB(nn.Module):
        def __init__(self): super().__init__(); self.d = 4; self.lm = None
        def forward(self, idx, valid, prefix=None): return idx[..., None].float().expand(-1, -1, 4)
    m = RRModel.__new__(RRModel); nn.Module.__init__(m); m.backbone, m.pool, m.prompt = _BB(), "mean", None
    idx = torch.tensor([[5, 7, 0, 9, 0]]); seg = torch.tensor([[0, 0, 0, 1, 1]])
    v = m.sentence_vectors(idx, torch.ones_like(idx, dtype=torch.bool), seg, 2)
    assert torch.allclose(v[:, 0], torch.tensor([4.0, 4.5])), v
    m.pool = "sep"; v = m.sentence_vectors(idx, torch.ones_like(idx, dtype=torch.bool), seg, 2)
    assert torch.allclose(v[:, 0], torch.tensor([0.0, 0.0])), v
    truth = {d["id"]: d["labels"] for d in DEV_DOCS}
    print(f"unit tests passed | gold vs gold official dev macro-F1 = {rr_scores(truth, truth)['macro_f1']:.4f}")
_tests()

unit tests passed | gold vs gold official dev macro-F1 = 0.8462


## 11. Run the queue (one worker thread per GPU)

In [12]:
def run_queue(names):
    q = _queue.Queue()
    for n in names: q.put(n)
    devices = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)] or [torch.device("cpu")]
    def worker(dev):
        while True:
            try: name = q.get_nowait()
            except _queue.Empty: return
            if time_left_h() < MIN_HOURS_PER_EXP: print(f"[{name}] skipped: {time_left_h():.2f} h left"); continue
            try:
                if dev.type == "cuda":
                    with torch.cuda.device(dev): run_experiment(name, dev)
                else: run_experiment(name, dev)
            except Exception:
                print(f"[{name}] FAILED:\n{traceback.format_exc()}", flush=True)
                (EXPS_DIR / name).mkdir(parents=True, exist_ok=True); (EXPS_DIR / name / "error.txt").write_text(traceback.format_exc())
    threads = [threading.Thread(target=worker, args=(dv,)) for dv in devices]
    for t in threads: t.start()
    for t in threads: t.join()

t_queue = time.time(); run_queue(RUN_QUEUE); print(f"queue finished in {(time.time() - t_queue)/60:.1f} min")

[f2_A1_seppool_lora | cuda:1] model A1 (modern) attn=causal method=lora ctx=window lstm=256 pos=True head=linear domains=('CL', 'IT') | params 114.9M, trainable 4.788M (4.17%) | 393 chunks ≈ 1728 windows/epoch, 3940 steps
[f4_B_kd | cuda:0] teacher b5_A1_causal_seppool (A1, 112.3M params): soft targets for 80 docs in 26s
[f4_B_kd | cuda:0] model B (encdec) attn=bidir method=full ctx=window lstm=256 pos=True head=linear domains=('CL', 'IT') | params 69.8M, trainable 69.778M (100.00%) | 393 chunks ≈ 1728 windows/epoch, 2364 steps
[f2_A1_seppool_lora | cuda:1] {"epoch": 1, "loss": 1.8658, "min": 1.0034, "dev_f1": 0.371}
[f4_B_kd | cuda:0] {"epoch": 1, "loss": 3.2034, "min": 0.5929, "dev_f1": 0.2563}
[f4_B_kd | cuda:0] {"epoch": 2, "loss": 1.8484, "min": 1.2638, "dev_f1": 0.3129}
[f2_A1_seppool_lora | cuda:1] {"epoch": 2, "loss": 1.1843, "min": 2.1522, "dev_f1": 0.4258}
[f4_B_kd | cuda:0] {"epoch": 3, "loss": 1.2817, "min": 1.9775, "dev_f1": 0.3471}
[f4_B_kd | cuda:0] {"epoch": 4, "loss": 

## 12. Results (this run + attached earlier runs)
`dev_macro_f1` = official metric (13 labels; ceiling on our dev = 11/13 = 0.846) after dev-tuned biases;
`dev_f1_raw` = before tuning; `dev_f1_present` = macro over the 11 labels that occur in dev; CL_f1 / IT_f1 per domain.

In [13]:
rows = [load_json(f)["row"] for f in sorted(EXPS_DIR.glob("*/result.json"))]
if rows:
    df = pd.DataFrame(rows).sort_values("dev_macro_f1", ascending=False)
    df.to_csv(ROOT / "p3" / "p3_experiments.csv", index=False)
    cols = ["name", "model", "attn", "method", "ctx", "lstm", "pos", "head", "pool", "domains", "dev_macro_f1", "dev_f1_raw",
            "dev_f1_present", "dev_acc", "CL_f1", "IT_f1", "trainable_params_M", "trainable_%", "peak_mem_GB", "train_min", "best_epoch"]
    display(df[[c for c in cols if c in df]].round(4).to_string(index=False))
    print("\nper-label dev F1:")
    display(df[["name"] + [f"F1_{l}" for l in LABELS]].round(3).to_string(index=False))
    best = df.iloc[0]["name"]; m = load_json(EXPS_DIR / best / "dev_metrics.json")
    print(f"\nbest: {best} -> {ROOT}/p3/exps/{best}/submission.zip")
    print("most frequent confusions (gold -> predicted):", m["confusions(gold->pred)"])
for f in sorted(EXPS_DIR.glob("*/error.txt")): print(f"FAILED: {f.parent.name}\n{f.read_text()[-1500:]}")

'                name model       attn                        method    ctx  lstm  pos   head pool domains  dev_macro_f1  dev_f1_raw  dev_f1_present  dev_acc  CL_f1  IT_f1  trainable_params_M  trainable_%  peak_mem_GB  train_min  best_epoch\n f6_A1_seppool_ctx96    A1     causal                          full window   256 True linear  sep   CL+IT        0.5374      0.5204          0.6351   0.6877 0.4384 0.5205            112.2537      100.000       5.9171    16.8650           6\nb5_A1_causal_seppool    A1     causal                          full window   256 True linear  sep   CL+IT        0.5145      0.4823          0.6081   0.6144 0.4679 0.5157            112.2537      100.000       6.1646    13.9308           4\n  f2_A1_seppool_lora    A1     causal                          lora window   256 True linear  sep   CL+IT        0.5086      0.4785          0.6010   0.6303 0.4788 0.4755              4.7882        4.167       4.3775    22.8007           7\n        b1_A0_causal    A0     caus


per-label dev F1:


'                name  F1_Fact  F1_Issue  F1_ArgumentPetitioner  F1_ArgumentRespondent  F1_Statute  F1_Dissent  F1_PrecedentReliedUpon  F1_PrecedentNotReliedUpon  F1_PrecedentOverruled  F1_RulingByLowerCourt  F1_RatioOfTheDecision  F1_RulingByPresentCourt  F1_None\n f6_A1_seppool_ctx96    0.742     0.714                  0.729                  0.724       0.815         0.0                   0.529                      0.376                    0.0                  0.667                  0.696                    0.637    0.357\nb5_A1_causal_seppool    0.654     0.667                  0.647                  0.579       0.830         0.0                   0.503                      0.439                    0.0                  0.723                  0.596                    0.659    0.392\n  f2_A1_seppool_lora    0.725     0.643                  0.587                  0.655       0.811         0.0                   0.556                      0.330                    0.0                  0.6


best: f6_A1_seppool_ctx96 -> /kaggle/working/cs781_p1/p3/exps/f6_A1_seppool_ctx96/submission.zip
most frequent confusions (gold -> predicted): {'ArgumentRespondent->ArgumentPetitioner': 63, 'PrecedentReliedUpon->RatioOfTheDecision': 57, 'PrecedentReliedUpon->Fact': 52, 'RatioOfTheDecision->PrecedentReliedUpon': 44, 'Fact->ArgumentPetitioner': 36, 'ArgumentPetitioner->PrecedentReliedUpon': 29, 'RatioOfTheDecision->Fact': 29, 'ArgumentPetitioner->RatioOfTheDecision': 28, 'ArgumentPetitioner->Fact': 27, 'Fact->RatioOfTheDecision': 26, 'Statute->RatioOfTheDecision': 23, 'Fact->RulingByLowerCourt': 17, 'RulingByLowerCourt->Fact': 17, 'RulingByPresentCourt->RatioOfTheDecision': 17, 'ArgumentRespondent->PrecedentReliedUpon': 16, 'None->PrecedentReliedUpon': 16, 'ArgumentPetitioner->ArgumentRespondent': 15, 'PrecedentNotReliedUpon->PrecedentReliedUpon': 15, 'ArgumentRespondent->Fact': 14, 'RulingByLowerCourt->RatioOfTheDecision': 12, 'ArgumentRespondent->PrecedentNotReliedUpon': 9, 'RatioOfTh